In [ ]:
from argparse import Namespace
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.build import build_dataset
from src.misc import load_config
from src.predict import AnchorSlice, MPDDOptions, load_predictor

In [ ]:
PREDICT_CONFIG = ROOT / "config" / "predict.yaml"
ANCHOR_AXIS = 0
ANCHOR_COUNT = 3
SEED = 0

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)

config = load_config(PREDICT_CONFIG)
run_dir = Path(config.pop("run_dir"))
run_dir = run_dir if run_dir.is_absolute() else ROOT / run_dir
options = MPDDOptions(**config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
predictor = load_predictor(run_dir, device=device)

data_args = Namespace(**load_config(run_dir / "model.yaml"))
data_dir = Path(data_args.data_dir)
data_args.data_dir = data_dir if data_dir.is_absolute() else ROOT / data_dir
data_args.augment = False
dataset = build_dataset(data_args)

positions = np.linspace(0, options.volume_size - 1, ANCHOR_COUNT + 2)[1:-1]
anchors = [
    AnchorSlice(
        image=dataset[index % len(dataset)][0][0].numpy().astype(np.uint8),
        axis=ANCHOR_AXIS,
        index=int(round(position)),
    )
    for index, position in enumerate(positions)
]

start = time.perf_counter()
volume, stats = predictor.predict(options, anchors=anchors)
elapsed = time.perf_counter() - start

sample_fractions = [round(value, 4) for value in stats["phase_fractions"]]
print(
    f"anchors={len(anchors)} voxels={stats['anchor_voxels']} "
    f"elapsed={elapsed:.1f}s fractions={sample_fractions}"
)

In [ ]:
fig, axes = plt.subplots(len(anchors), 3, figsize=(10, 3 * len(anchors)), squeeze=False)
for row, anchor in enumerate(anchors):
    generated = volume.select(anchor.axis, anchor.index).numpy()
    panels = (
        (anchor.image, "anchor", "gray", 0, options.num_phases - 1),
        (generated, "generated", "gray", 0, options.num_phases - 1),
        (generated != anchor.image, "different", "magma", 0, 1),
    )
    for axis, (image, title, cmap, vmin, vmax) in zip(axes[row], panels):
        axis.imshow(
            image,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            interpolation="nearest",
        )
        axis.set_title(f"{title} / index {anchor.index}")
        axis.axis("off")
plt.tight_layout()